# RAGNN Self-Loop Usage Diagnostic

Measures how much the RAGNN relies on self-loop paths (vs latent neighbour edges) at each message-passing layer during a live agent rollout.

**Stat definition** (`self_loop_usage_l` in `RAGNN.stats`):
```
self_loop_usage_l = ‖h_new_selfloops − h‖ / ‖h_new − h‖
```
where `h_new_selfloops` is the layer output when all latent edge weights are zeroed.
- **→ 0**: latent edges dominate — neighbours carry meaningful signal.
- **→ 1**: decoder collapse — model ignores discovered edges; self-loops carry all the information.

**Workflow:**
1. Find best checkpoint per seed in the experiment directory.
2. For each seed: roll out one episode at a time, collect `self_loop_usage_l` every time the RL model is invoked (`rho_max > rho_thresh`).
3. Plot box/strip distributions per layer and per seed.

In [1]:
%matplotlib inline
import logging
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

logging.basicConfig(level=logging.WARNING)
plt.style.use("default")

REPO_ROOT = Path("/home/adrian/Dev/NRI-for-explainable-RL-in-Power-Grids").resolve()
os.chdir(REPO_ROOT)
for p in [str(REPO_ROOT / "src"), str(REPO_ROOT / "experiments"), str(REPO_ROOT)]:
    if p not in sys.path:
        sys.path.insert(0, p)

from analysis.cross_seed_analysis import find_trial_dirs, get_seed, best_checkpoint, load_agent

In [2]:
# ── experiment ─────────────────────────────────────────────────────────────────
EXPERIMENT_DIR  = REPO_ROOT / "results/2026_07_20_IEEE14/rappo_multiseed_gcnconv"
OUT_DIR         = EXPERIMENT_DIR / "cross_seed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── rollout settings ───────────────────────────────────────────────────────────
ENV_NAME            = "l2rpn_case14_sandbox_test"
RHO_THRESH          = 0.95   # model-activation threshold (same as training)
N_ACTIVATIONS       = 200    # target model invocations to collect per seed
MAX_CHRONICS        = 100    # upper bound on episodes per seed
SHUFFLE_SEED        = 42

# ── checkpoint ─────────────────────────────────────────────────────────────────
CONV_TYPE       = None   # None → from checkpoint config
CHECKPOINT_NAME = None   # None → best per trial from progress.csv

In [3]:
import ray

ray.init(
    ignore_reinit_error=True,
    logging_level=logging.ERROR,
    log_to_driver=False,
    num_cpus=1,
    object_store_memory=512 * 1024 * 1024,
)

trial_dirs = find_trial_dirs(EXPERIMENT_DIR)
print(f"Found {len(trial_dirs)} trial dir(s) in {EXPERIMENT_DIR}")

seed_to_info: dict[int, tuple[Path, str]] = {}
for td in trial_dirs:
    try:
        seed = get_seed(td)
    except KeyError as e:
        print(f"  WARNING: skipping {td.name} — {e}")
        continue
    try:
        ckpt = CHECKPOINT_NAME or best_checkpoint(td)
    except FileNotFoundError as e:
        print(f"  WARNING: skipping {td.name} — {e}")
        continue
    seed_to_info[seed] = (td, ckpt)
    print(f"  seed={seed}  checkpoint={ckpt}  trial={td.name}")

seeds = sorted(seed_to_info.keys())
print(f"\nSeeds: {seeds}")

/home/adrian/.conda/envs/L2RPN/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-07-23 15:28:06,723	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


Found 5 trial dir(s) in /home/adrian/Dev/NRI-for-explainable-RL-in-Power-Grids/results/2026_07_20_IEEE14/rappo_multiseed_gcnconv
  seed=0  checkpoint=checkpoint_000008  trial=CustomPPO_RARL_5887369_1b111_2026-07-18_21-30-16
  seed=1  checkpoint=checkpoint_000007  trial=CustomPPO_RARL_5887370_1ae2e_2026-07-18_21-30-16
  seed=2  checkpoint=checkpoint_000007  trial=CustomPPO_RARL_5887371_15dd9_2026-07-18_21-30-08
  seed=3  checkpoint=checkpoint_000007  trial=CustomPPO_RARL_5887372_1a964_2026-07-18_21-30-16
  seed=4  checkpoint=checkpoint_000006  trial=CustomPPO_RARL_5887373_1c693_2026-07-18_21-30-19

Seeds: [0, 1, 2, 3, 4]


## Rollout & Stats Collection

For each seed we load the agent and roll out episodes manually so we can inspect
`RAGNN.stats` immediately after every model invocation.

In [4]:
def rollout_and_collect_stats(
    agent,
    g2op_env,
    n_activations: int = 200,
    max_chronics: int = 100,
    shuffle_seed: int = 42,
) -> list[dict]:
    """
    Roll out the agent and collect RAGNN self-loop usage stats.

    Every time the RL model is invoked (``rho_max > activation_thresh``),
    all ``self_loop_usage_<l>`` entries in ``RAGNN.stats`` are recorded.

    :param agent: Loaded RllibAgent.
    :param g2op_env: Underlying Grid2Op environment.
    :param n_activations: Target number of model invocations to record.
    :param max_chronics: Maximum number of episodes to attempt.
    :param shuffle_seed: RNG seed for chronic order shuffling.
    :return: List of dicts with keys layer, value, rho_max, chronic_id, step.
    """
    model = agent._rllib_agent.model
    model.eval()   # ensure eval mode (BatchNorm uses running stats)

    n_chronics = len(g2op_env.chronics_handler.subpaths)
    chronic_ids = list(range(n_chronics))
    np.random.default_rng(shuffle_seed).shuffle(chronic_ids)

    records: list[dict] = []
    activation_count = 0

    for chronic_id in chronic_ids[:max_chronics]:
        if activation_count >= n_activations:
            break

        g2op_env.set_id(chronic_id)
        obs = g2op_env.reset()
        reward, done = 0.0, False
        step = 0

        while not done:
            action = agent.activation_function(obs, reward, done)

            # agent.rho_max is set inside act() for the current obs.
            # If it exceeds the threshold, compute_single_action was called
            # and RAGNN.stats was freshly populated.
            if agent.rho_max > agent.activation_thresh:
                gnn_stats = model.ragnn.gnn.stats
                rho_max = float(obs.rho.max())
                for key, val in gnn_stats.items():
                    if key.startswith("self_loop_usage_"):
                        layer = int(key.split("_")[-1])
                        records.append({
                            "layer": layer,
                            "value": float(val.item() if hasattr(val, "item") else val),
                            "rho_max": rho_max,
                            "chronic_id": chronic_id,
                            "step": step,
                        })
                activation_count += 1
                if activation_count >= n_activations:
                    break

            obs, reward, done, _ = g2op_env.step(action)
            step += 1

    print(
        f"  Collected {activation_count} model activations "
        f"over {chronic_id + 1} episode(s)  →  {len(records)} stat entries"
    )
    return records

In [5]:
all_records: list[dict] = []
gym_wrappers = []  # keep alive to prevent premature cleanup

for seed in seeds:
    trial_dir, ckpt_name = seed_to_info[seed]
    print(f"Seed {seed}: loading {ckpt_name} from {trial_dir.name} ...")

    agent, g2op_env, gym_wrapper = load_agent(
        trial_dir, ckpt_name, ENV_NAME, conv_type=CONV_TYPE
    )
    gym_wrappers.append(gym_wrapper)

    records = rollout_and_collect_stats(
        agent, g2op_env,
        n_activations=N_ACTIVATIONS,
        max_chronics=MAX_CHRONICS,
        shuffle_seed=SHUFFLE_SEED,
    )
    for r in records:
        r["seed"] = seed
    all_records.extend(records)

df = pd.DataFrame(all_records)
print(f"\nTotal records: {len(df)}")
print(df.groupby(["seed", "layer"])["value"].describe().round(3))

Seed 0: loading checkpoint_000008 from CustomPPO_RARL_5887369_1b111_2026-07-18_21-30-16 ...


/home/adrian/.conda/envs/L2RPN/lib/python3.10/site-packages/grid2op/MakeEnv/Make.py:13: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


  Collected 200 model activations over 29 episode(s)  →  600 stat entries
Seed 1: loading checkpoint_000007 from CustomPPO_RARL_5887370_1ae2e_2026-07-18_21-30-16 ...
  Collected 200 model activations over 39 episode(s)  →  600 stat entries
Seed 2: loading checkpoint_000007 from CustomPPO_RARL_5887371_15dd9_2026-07-18_21-30-08 ...
  Collected 200 model activations over 24 episode(s)  →  600 stat entries
Seed 3: loading checkpoint_000007 from CustomPPO_RARL_5887372_1a964_2026-07-18_21-30-16 ...
  Collected 200 model activations over 8 episode(s)  →  600 stat entries
Seed 4: loading checkpoint_000006 from CustomPPO_RARL_5887373_1c693_2026-07-18_21-30-19 ...
  Collected 200 model activations over 5 episode(s)  →  600 stat entries

Total records: 3000
            count   mean    std    min    25%    50%    75%    max
seed layer                                                        
0    0      200.0  1.199  0.043  1.077  1.192  1.216  1.227  1.250
     1      200.0  1.074  0.023  1.016  1.

## Visualisation

Each box shows the distribution of `self_loop_usage_l` over all model invocations for one (seed, layer) pair.

- **y → 0** : latent edges dominate — good.
- **y → 1** : decoder collapse — model relies on self-loops only.

In [6]:
layers = sorted(df["layer"].unique())
n_layers = len(layers)
n_seeds = len(seeds)

# ── colour palette ─────────────────────────────────────────────────────────────
palette = plt.cm.tab10.colors
seed_colours = {s: palette[i % len(palette)] for i, s in enumerate(seeds)}

# ── figure: one subplot per layer, boxplot per seed ────────────────────────────
fig, axes = plt.subplots(
    1, n_layers,
    figsize=(4 * n_layers, 5),
    sharey=True,
)
if n_layers == 1:
    axes = [axes]

for ax, layer in zip(axes, layers):
    data_per_seed = [
        df[(df["layer"] == layer) & (df["seed"] == s)]["value"].values
        for s in seeds
    ]
    bp = ax.boxplot(
        data_per_seed,
        patch_artist=True,
        widths=0.5,
        medianprops=dict(color="black", linewidth=2),
        flierprops=dict(marker=".", markersize=3, alpha=0.4),
    )
    for patch, seed in zip(bp["boxes"], seeds):
        patch.set_facecolor(seed_colours[seed])
        patch.set_alpha(0.7)

    ax.axhline(1.0, color="red",  linewidth=1.2, linestyle="--", label="collapse (1)")
    ax.axhline(0.0, color="green", linewidth=1.2, linestyle="--", label="edge-driven (0)")
    ax.set_xticks(range(1, n_seeds + 1))
    ax.set_xticklabels([f"seed {s}" for s in seeds], rotation=30, ha="right")
    ax.set_title(f"Layer {layer}", fontsize=12)
    ax.set_ylim(-0.05, 1.1)
    ax.grid(axis="y", alpha=0.3)

axes[0].set_ylabel("self_loop_usage", fontsize=11)
fig.suptitle(
    "RAGNN self-loop usage per layer\n"
    "(measured at model invocations where rho > {:.2f})".format(RHO_THRESH),
    fontsize=13,
)
plt.tight_layout()

out_path = OUT_DIR / "self_loop_usage_boxplot.png"
fig.savefig(out_path, dpi=150, bbox_inches="tight")
print(f"Saved: {out_path}")
plt.show()

Saved: /home/adrian/Dev/NRI-for-explainable-RL-in-Power-Grids/results/2026_07_20_IEEE14/rappo_multiseed_gcnconv/cross_seed/self_loop_usage_boxplot.png


/tmp/ipykernel_46852/4095419009.py:53: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# ── scatter: rho_max vs self_loop_usage, faceted by layer ─────────────────────
fig, axes = plt.subplots(
    1, n_layers,
    figsize=(4.5 * n_layers, 4),
    sharey=True,
)
if n_layers == 1:
    axes = [axes]

for ax, layer in zip(axes, layers):
    sub = df[df["layer"] == layer]
    for seed in seeds:
        mask = sub["seed"] == seed
        ax.scatter(
            sub.loc[mask, "rho_max"],
            sub.loc[mask, "value"],
            s=10, alpha=0.4,
            color=seed_colours[seed],
            label=f"seed {seed}",
        )
    ax.axhline(1.0, color="red",   linewidth=1, linestyle="--")
    ax.axhline(0.0, color="green", linewidth=1, linestyle="--")
    ax.set_xlabel("rho_max", fontsize=10)
    ax.set_title(f"Layer {layer}", fontsize=12)
    ax.set_ylim(-0.05, 1.1)
    ax.grid(alpha=0.3)

axes[0].set_ylabel("self_loop_usage", fontsize=11)
axes[-1].legend(loc="upper right", fontsize=8, markerscale=2)
fig.suptitle(
    "rho_max vs self-loop usage per layer",
    fontsize=13,
)
plt.tight_layout()

out_path = OUT_DIR / "self_loop_usage_scatter.png"
fig.savefig(out_path, dpi=150, bbox_inches="tight")
print(f"Saved: {out_path}")
plt.show()

Saved: /home/adrian/Dev/NRI-for-explainable-RL-in-Power-Grids/results/2026_07_20_IEEE14/rappo_multiseed_gcnconv/cross_seed/self_loop_usage_scatter.png


/tmp/ipykernel_46852/3618601362.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# ── numeric summary ────────────────────────────────────────────────────────────
summary = (
    df.groupby(["seed", "layer"])["value"]
    .agg(["mean", "median", "std", "min", "max"])
    .round(4)
)
print(summary)

              mean  median     std     min     max
seed layer                                        
0    0      1.1994  1.2163  0.0432  1.0765  1.2500
     1      1.0736  1.0825  0.0230  1.0163  1.1052
     2      1.0322  1.0344  0.0094  1.0092  1.0491
1    0      1.1615  1.1717  0.0413  1.0677  1.2465
     1      1.0674  1.0727  0.0197  1.0168  1.1002
     2      1.0367  1.0388  0.0125  1.0109  1.0623
2    0      1.3880  1.3840  0.0393  1.1994  1.5132
     1      1.1468  1.1526  0.0227  1.0288  1.1807
     2      1.0937  1.0967  0.0132  1.0318  1.1168
3    0      1.1138  1.1032  0.0358  1.0786  1.2528
     1      1.0453  1.0398  0.0167  1.0271  1.1145
     2      1.0293  1.0262  0.0095  1.0170  1.0629
4    0      1.3047  1.3201  0.0483  1.1150  1.3561
     1      1.1440  1.1547  0.0320  1.0120  1.1715
     2      1.0694  1.0737  0.0145  1.0123  1.0881


In [9]:
ray.shutdown()